# 01. Exploratory data analysis

Distributions, sticky-vs-non-sticky comparisons, correlations, per-genre popularity, missingness, and a threshold-sensitivity table for the `sticky_top_q` label at q in {0.05, 0.10, 0.20, 0.30}. Inputs: `data/processed/tracks.parquet` built in Phase 1.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from audio_priors.labels import sticky_top_q

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FIG = ROOT / "outputs" / "figures"
TBL = ROOT / "outputs" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TBL.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

df = pd.read_parquet(ROOT / "data" / "processed" / "tracks.parquet")
print(f"rows: {len(df):,}, cols: {len(df.columns)}")
print(f"sources: {df['source'].value_counts().to_dict()}")

## 1. Missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
print(miss.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4.5))
miss.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Fraction missing")
ax.set_title("Missingness by column")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIG / "01_missingness.png", bbox_inches="tight")
plt.show()

Most missingness lives in `popularity` and `genre`, both of which the rodolfofigueroa source does not carry. Audio features are present throughout.

## 2. Popularity distribution

In [ ]:
pop = df["popularity"].dropna()
print(f"popularity-bearing rows: {len(pop):,}")
print(pop.describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(pop, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Popularity histogram")
axes[0].set_xlabel("popularity")
sns.kdeplot(pop, ax=axes[1], fill=True, color="steelblue", alpha=0.5)
axes[1].set_title("Popularity KDE")
axes[1].set_xlabel("popularity")
fig.tight_layout()
fig.savefig(FIG / "02_popularity_distribution.png", bbox_inches="tight")
plt.show()

## 3. Audio features by sticky label (q = 0.20)

In [ ]:
labeled = df.dropna(subset=["popularity"]).copy()
labeled["sticky"] = sticky_top_q(labeled, q=0.20)
print(f"labeled rows: {len(labeled):,}, sticky rate: {labeled['sticky'].mean():.3f}")

feature_cols = [
    "danceability", "energy", "valence", "acousticness",
    "instrumentalness", "liveness", "speechiness",
]
fig, axes = plt.subplots(2, 4, figsize=(14, 7), squeeze=False)
for i, col in enumerate(feature_cols):
    r, c = divmod(i, 4)
    sns.boxplot(
        data=labeled,
        x="sticky", y=col,
        ax=axes[r][c],
        hue="sticky", palette="Set2", legend=False,
    )
    axes[r][c].set_title(col)
    axes[r][c].set_xlabel("sticky (0/1)")
axes[1][3].set_visible(False)
fig.suptitle("Audio features by sticky label (q = 0.20)", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "03_features_by_sticky.png", bbox_inches="tight")
plt.show()

## 4. Correlation heatmap

In [ ]:
corr_cols = ["popularity"] + feature_cols + ["loudness", "tempo", "duration_ms"]
corr = labeled[corr_cols].corr()
print(corr["popularity"].sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Correlation of popularity and audio features")
fig.tight_layout()
fig.savefig(FIG / "04_correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 5. Per-genre popularity

In [ ]:
with_genre = labeled.dropna(subset=["genre"])
genre_stats = (
    with_genre.groupby("genre")["popularity"]
    .agg(["count", "mean", "median"])
    .sort_values("count", ascending=False)
    .head(20)
)
print(genre_stats.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 6))
genre_stats["mean"].sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Mean popularity")
ax.set_title("Top 20 genres by row count, mean popularity")
fig.tight_layout()
fig.savefig(FIG / "05_popularity_by_genre.png", bbox_inches="tight")
plt.show()

## 6. Threshold sensitivity

Logistic regression on standardized audio features at four candidate q values. 20% stratified hold-out, `random_state=42`. Reports test ROC-AUC and the class balance produced by each q.

In [ ]:
X_full = labeled[feature_cols + ["loudness", "tempo", "duration_ms"]].dropna()
y_pop = labeled.loc[X_full.index, "popularity"]

rows = []
for q in [0.05, 0.10, 0.20, 0.30]:
    threshold = float(y_pop.quantile(1.0 - q))
    y = (y_pop >= threshold).astype(int)
    balance = float(y.mean())
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y, test_size=0.20, stratify=y, random_state=42
    )
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
    ])
    pipe.fit(X_train, y_train)
    score = pipe.predict_proba(X_test)[:, 1]
    auc = float(roc_auc_score(y_test, score))
    rows.append({
        "q": q,
        "threshold": threshold,
        "class_balance": balance,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "roc_auc": auc,
    })

table = pd.DataFrame(rows)
print(table.round(4).to_string(index=False))
table.to_csv(TBL / "label_sensitivity.csv", index=False)
print(f"wrote {TBL / 'label_sensitivity.csv'}")

## Notes

1. The label sensitivity table is the source of truth for choosing q
   downstream. Phase 3 picks q from this table, justified by the AUC
   plus class balance trade-off.
2. Per-genre AUC is computed in Phase 3 on the chosen q. EDA here
   stops at the global view because the small genres in the corpus
   have too few rows for stable group-level AUCs.
3. Audio-feature correlations with popularity are small in absolute
   magnitude. This is the signal-thinness finding the rebuild needs
   to defend through bootstrap CIs in Phase 3.